In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ==========================================
# 1. USER INPUTS & PARAMETERS
# ==========================================
np.random.seed(42)  
n_paths = 10000        # Number of Monte Carlo simulations
n_years = 5            # 5-year forecast
n_quarters = n_years * 4 
dt = 0.25              # Quarterly time step
S0 = 4677              # today's gold price in $/oz          

# Replace these arrays with your actual quarterly data (Length must be exactly 20)
# Production in oz per quarter
quarterly_production = np.array([
    704, 741, 752, 751,             #2026
    720, 743, 748, 814,             #2027
    783, 787, 789, 811,             #2028
    805, 821, 842, 814,             #2029
    802, 782, 769, 779]) * 1000     #2030

# Absolute Total Costs in Dollars/oz per quarter
quarterly_costs = np.array([
    1685.07, 1698.87, 1727.07, 1783.58,     #2026
    1820.64, 1856.45, 1899.48, 1945.25,     #2027
    1986.89, 2031.28, 2077.64, 2124.60,     #2028
    2171.84, 2220.87, 2270.94, 2321.98,     #2029
    2374.12, 2427.60, 2482.20, 2538.02      #2030
])

# ==========================================
# 2. EXTRACTING SIGMA FROM CSV
# ==========================================
# Load the options chain data
options_data = pd.read_csv('gld_chain_with_iv.csv')

# use Implied Volatility for different quarters
sigma_curve = np.array([
    0.357, 0.330, 0.307, 0.311, 0.304, 0.301, 0.301, 0.301, 0.297, 0.297, 
    0.293, 0.290, 0.290, 0.290, 0.290, 0.290, 0.290, 0.290, 0.290, 0.290
])

#utilizzare 0,29 - 0,33 - 0,35 e mandare avanti i tre scenari tenendola costante nei 5 anni

# =========================================
# 3. SIEGEL-SVENSSON YIELD CURVE (DRIFT)
# =========================================
# Exact parameters from the Svensson curve
beta0 = 0.0557186309095699
beta1 = -0.01857773468290351
beta2 = -0.010955973627381332
beta3 = -0.019216770175974365
tau1 = 2.0
tau2 = 5.0

# Generate exact time steps in years (0.25, 0.50 ... 5.00)
time_steps = np.arange(1, n_quarters + 1) * dt

# Calculate the time-varying yield vector (r_curve)
term2 = (1 - np.exp(-time_steps / tau1)) / (time_steps / tau1)
term3 = term2 - np.exp(-time_steps / tau1)
term4 = ((1 - np.exp(-time_steps / tau2)) / (time_steps / tau2)) - np.exp(-time_steps / tau2)

r_curve = beta0 + (beta1 * term2) + (beta2 * term3) + (beta3 * term4)

# ==========================================
# 4. GEOMETRIC BROWNIAN MOTION ENGINE
# ==========================================
# Generate random standard normal shocks
Z = np.random.standard_normal((n_paths, n_quarters))

# Initialize the price matrix
prices = np.zeros_like(Z)
prices[:, 0] = S0

# Simulate exactly across time using the dynamic r_curve
for t in range(1, n_quarters):
    r_t = r_curve[t]
    sigma_star = sigma_curve[t]
    prices[:, t] = prices[:, t-1] * np.exp((r_t - 0.5 * sigma_star**2) * dt + sigma_star * np.sqrt(dt) * Z[:, t])

# ==========================================
# 5. EBITDA CALCULATION & ANNUAL AGGREGATION
# ==========================================
# Broadcast production and costs across all 10,000 price paths
prod_matrix = quarterly_production.reshape(1, -1)
cost_matrix = quarterly_costs.reshape(1, -1)

# Calculate Quarterly EBITDA Matrix: (Price * Production) - Costs
quarterly_ebitda_matrix = (prices * prod_matrix) - cost_matrix * prod_matrix

# Aggregate Quarterly to Annual EBITDA
# We reshape the (10000, 20) matrix into a 3D matrix of (10000, 5, 4) 
# and sum across the 3rd axis (the 4 quarters) to get exact annual figures.
annual_ebitda_matrix = quarterly_ebitda_matrix.reshape(n_paths, n_years, 4).sum(axis=2)

# ==========================================
# 6. OUTPUT PERCENTILES
# ==========================================
# Extract the 10th, 50th (Median), and 90th percentiles for the annual data
annual_lower = np.percentile(annual_ebitda_matrix, 10, axis=0)
annual_median = np.percentile(annual_ebitda_matrix, 50, axis=0)
annual_upper = np.percentile(annual_ebitda_matrix, 90, axis=0)

# Format into a clean DataFrame
annual_forecast = pd.DataFrame({
    'Year': range(1, n_years + 1),
    'Annual_EBITDA_10th_CI': annual_lower,
    'Annual_EBITDA_Median': annual_median,
    'Annual_EBITDA_90th_CI': annual_upper
})

# Display the final results, converted to standard integers for readability
print("Forecasted Annual EBITDA (in Dollars):")
print("-" * 70)
print(annual_forecast.round(0).astype(int).map(lambda x: f"{x:,}").to_string(index=False))

In [ ]:
# ==========================================
# 6. EXTRACTING COMPREHENSIVE STATISTICS
# ==========================================
# Calculate a full suite of stats across all 10,000 simulated realities
stats_df = pd.DataFrame({
    'Year': range(1, n_years + 1),
    'Mean': np.mean(annual_ebitda_matrix, axis=0),
    'Std_Dev': np.std(annual_ebitda_matrix, axis=0),
    'Min_WorstCase': np.min(annual_ebitda_matrix, axis=0),
    '10th_Pct': np.percentile(annual_ebitda_matrix, 10, axis=0),
    'Median': np.median(annual_ebitda_matrix, axis=0),
    '90th_Pct': np.percentile(annual_ebitda_matrix, 90, axis=0),
    'Max_BestCase': np.max(annual_ebitda_matrix, axis=0)
})

print("Comprehensive Annual EBITDA Statistics (in Billions):")
print("-" * 120) # Extended the dashed line to match the wider table

# Format the entire table into Billions (e.g. $8.54B) for clean reading
formatted_stats = stats_df.copy()
for col in formatted_stats.columns[1:]:
    formatted_stats[col] = formatted_stats[col].apply(lambda x: f"${x*1e-9:,.2f}B")

# The exact fix: col_space forces the width, justify aligns the headers perfectly
print(formatted_stats.to_string(index=False, col_space=14, justify='right'))

# ==========================================
# 7. PLOTTING THE INSTITUTIONAL FAN CHART
# ==========================================
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

plt.figure(figsize=(8, 4))
years = np.arange(1, n_years + 1)

# Calculate exact percentiles to draw the shaded confidence bands
pct_5 = np.percentile(annual_ebitda_matrix, 5, axis=0)
pct_25 = np.percentile(annual_ebitda_matrix, 25, axis=0)
pct_50 = np.median(annual_ebitda_matrix, axis=0)
pct_75 = np.percentile(annual_ebitda_matrix, 75, axis=0)
pct_95 = np.percentile(annual_ebitda_matrix, 95, axis=0)

# 1. Plot the Light Blue Band (90% Confidence Interval)
plt.fill_between(years, pct_5, pct_95, color='royalblue', alpha=0.15, label='90% Confidence (5th - 95th Pct)')

# 2. Plot the Dark Blue Band (50% Confidence Interval)
plt.fill_between(years, pct_25, pct_75, color='royalblue', alpha=0.4, label='50% Core Probability (25th - 75th Pct)')

# 3. Plot the Median Line
plt.plot(years, pct_50, color='darkred', linewidth=2.5, label='Median Expected EBITDA')

# Optional: Plot just 5 "ghost" lines to show the jagged reality of a single path
plt.plot(years, annual_ebitda_matrix[:5, :].T, color='black', alpha=0.15, linewidth=1)

plt.title('Monte Carlo Simulation of Annual EBITDA (Confidence Fan Chart)')
plt.xlabel('Years')
plt.ylabel('Annual EBITDA (USD)')
plt.xticks(years, labels=['Year 1', 'Year 2', 'Year 3', 'Year 4', 'Year 5'])

# Format Y-Axis into Billions
def format_billions(x, pos):
    return f'${x*1e-9:,.1f}B'
plt.gca().yaxis.set_major_formatter(FuncFormatter(format_billions))

plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

In [ ]:
# ==========================================
# 7. PLOTTING THE MONTE CARLO PATHS
# ==========================================
plt.figure(figsize=(20, 10))

# Plot the first 200 ANNUAL paths
plt.plot(np.arange(1, n_years + 1), annual_ebitda_matrix[:1000, :].T, color='blue', alpha=0.05)
plt.plot(np.arange(1, n_years + 1), np.median(annual_ebitda_matrix, axis=0), color='red', linewidth=2, label='Median Annual EBITDA Path')

plt.title('Monte Carlo Simulation of Annual EBITDA')
plt.xlabel('Years')
plt.ylabel('Annual EBITDA')
plt.xticks(np.arange(1, n_years + 1), labels=['Year 1', 'Year 2', 'Year 3', 'Year 4', 'Year 5'])

# Format the Y-axis to show clean Billions of dollars (e.g., $8.0B)
def format_billions(x, pos):
    return f'${x*1e-9:,.1f}B'
plt.gca().yaxis.set_major_formatter(FuncFormatter(format_billions))

plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()